# Unsupervised Machine Learning Project
## Customer Behavior Segmentation + Dimensionality Reduction + Anomaly Detection

**Dataset:** `Customer_Behavior_Unsupervised_Dataset.csv`

Is project mein aap ek hi dataset par ye topics cover karain gy

- Feature Extraction & Engineering
- PCA
- t-SNE
- K-Means Clustering
- Hierarchical Clustering
- DBSCAN
- Anomaly Detection
- Cluster Profiling & Business Insights

In [1]:
# Install if needed:
# pip install pandas numpy matplotlib scikit-learn scipy openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset

In [2]:
df = pd.read_csv('Customer_Behavior_Unsupervised_Dataset.csv')
df.head()

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5


In [3]:
print(df.shape)
df.info()

(1000, 12)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Customer_ID          1000 non-null   str    
 1   Age                  1000 non-null   int64  
 2   Gender               1000 non-null   str    
 3   City                 1000 non-null   str    
 4   Annual_Income        1000 non-null   int64  
 5   Monthly_Spending     1000 non-null   int64  
 6   Visits_Per_Month     1000 non-null   int64  
 7   Online_Orders        1000 non-null   int64  
 8   Avg_Order_Value      1000 non-null   int64  
 9   Time_on_Website_Min  1000 non-null   int64  
 10  Discount_Usage       1000 non-null   float64
 11  Satisfaction_Score   1000 non-null   float64
dtypes: float64(2), int64(7), str(3)
memory usage: 111.4 KB


## 2. Basic Data Cleaning Check

In [4]:
print('Missing values:')
print(df.isnull().sum())

Missing values:
Customer_ID            0
Age                    0
Gender                 0
City                   0
Annual_Income          0
Monthly_Spending       0
Visits_Per_Month       0
Online_Orders          0
Avg_Order_Value        0
Time_on_Website_Min    0
Discount_Usage         0
Satisfaction_Score     0
dtype: int64


In [5]:
print('Duplicate rows:', df.duplicated().sum())

Duplicate rows: 0


In [6]:
df.describe()

,Age,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,41.311000,167926.746000,45111.414000,7.411000,6.191000,8497.485000,156.042000,0.502500,3.742900
std,14.286791,78050.778205,37092.037904,3.060628,3.420266,13609.923576,85.604266,0.286481,0.713455
min,18.000000,30629.000000,500.000000,1.000000,0.000000,522.000000,5.000000,0.000000,2.500000
25%,28.000000,100633.000000,15684.000000,5.000000,4.000000,3981.500000,80.750000,0.250000,3.100000
50%,41.000000,171250.000000,35833.000000,7.000000,6.000000,7479.500000,157.000000,0.500000,3.700000
75%,54.000000,234854.000000,65345.750000,9.250000,9.000000,11191.750000,229.000000,0.750000,4.400000
max,65.000000,299997.000000,233160.000000,16.000000,17.000000,233160.000000,300.000000,1.000000,5.000000


## 3. Feature Engineering

Yahan raw data se meaningful features create kiye jayenge.

Examples:
- `Spending_Ratio`: income ke comparison mein spending
- `Avg_Spend_Per_Visit`: har visit par average spending
- `Online_Order_Ratio`: visits ke comparison mein online orders
- `Customer_Value`: online order value ka estimated value
- `Engagement_Score`: website time + visits + online orders ka combined score

In [7]:
df

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5
...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8
996,C0997,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8
997,C0998,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9
998,C0999,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6


In [8]:
df_fe = df.copy()
df_fe

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5
...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8
996,C0997,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8
997,C0998,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9
998,C0999,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6


In [9]:
df_fe.columns

Index(['Customer_ID', 'Age', 'Gender', 'City', 'Annual_Income',
       'Monthly_Spending', 'Visits_Per_Month', 'Online_Orders',
       'Avg_Order_Value', 'Time_on_Website_Min', 'Discount_Usage',
       'Satisfaction_Score'],
      dtype='str')

In [10]:
df_fe['Spending_Ratio'] = df_fe['Monthly_Spending'] / df_fe['Annual_Income']
df_fe

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6,0.104498
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6,0.154454
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7,0.359505
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1,0.001846
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5,0.061268
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8,0.899998
996,C0997,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8,0.824816
997,C0998,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9,0.241858
998,C0999,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6,0.021901


In [11]:
df_fe.head()

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6,0.104498
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6,0.154454
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7,0.359505
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1,0.001846
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5,0.061268


In [12]:
df_fe.tail()

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio
995,C0996,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8,0.899998
996,C0997,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8,0.824816
997,C0998,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9,0.241858
998,C0999,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6,0.021901
999,C1000,63,Female,Faisalabad,83946,500,3,0,5605,115,0.95,3.7,0.005956


In [13]:
df_fe.Visits_Per_Month.unique()

array([ 8,  7,  1,  5, 11,  6, 10, 14,  9, 12,  3,  4, 16,  2, 13, 15])

Agar `Visits_Per_Month` mein kisi customer ki value 0 hai, to usay temporary 1 se replace kar do, taake division error na aaye.

In [14]:
df_fe['Avg_Spend_Per_Visit'] = df_fe['Monthly_Spending'] / df_fe['Visits_Per_Month'].replace(0, 1)
df_fe

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio,Avg_Spend_Per_Visit
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6,0.104498,2275.375000
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6,0.154454,3353.428571
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7,0.359505,12920.142857
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1,0.001846,500.000000
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5,0.061268,662.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8,0.899998,4184.000000
996,C0997,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8,0.824816,9381.250000
997,C0998,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9,0.241858,3103.250000
998,C0999,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6,0.021901,6350.000000


In [15]:
df_fe['Online_Order_Ratio'] = df_fe['Online_Orders'] / df_fe['Visits_Per_Month'].replace(0, 1)
df_fe

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio,Avg_Spend_Per_Visit,Online_Order_Ratio
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6,0.104498,2275.375000,1.000000
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6,0.154454,3353.428571,0.857143
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7,0.359505,12920.142857,1.000000
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1,0.001846,500.000000,0.000000
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5,0.061268,662.000000,0.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8,0.899998,4184.000000,1.000000
996,C0997,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8,0.824816,9381.250000,0.750000
997,C0998,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9,0.241858,3103.250000,1.250000
998,C0999,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6,0.021901,6350.000000,2.000000


In [16]:
df_fe['Customer_Value'] = df_fe['Avg_Order_Value'] * df_fe['Online_Orders']
df_fe

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio,Avg_Spend_Per_Visit,Online_Order_Ratio,Customer_Value
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6,0.104498,2275.375000,1.000000,17432
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6,0.154454,3353.428571,0.857143,22542
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7,0.359505,12920.142857,1.000000,91063
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1,0.001846,500.000000,0.000000,0
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5,0.061268,662.000000,0.200000,1807
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8,0.899998,4184.000000,1.000000,61810
996,C0997,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8,0.824816,9381.250000,0.750000,38901
997,C0998,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9,0.241858,3103.250000,1.250000,25730
998,C0999,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6,0.021901,6350.000000,2.000000,7326


## Engagement Score hota kya hai?

Company ye dekhna chahti hai ke customer kitna active (engaged) hai.

Is activity ko measure karne ke liye hum ne 3 cheezen use ki hain:

`Visits_Per_Month`
`Online_Orders`
`Time_on_Website_Min`

Lekin ye tino equally important nahi hain.

Isliye hum ne weights diye hain.

`Visits_Per_Month     → 40%`
`Online_Orders        → 40%`
`Time_on_Website_Min  → 20%`

### Ye 0.4 aur 0.2 kahan se aaye?

Ye fixed values nahi hain.

Ye sirf assumption (business rule) hai.

`Visits × 40%`

`Orders × 40%`

`Time × 20%`

Engagement =

10 × 0.4 = 4

8 × 0.4 = 3.2

150 × 0.2 = 30

Final

4 + 3.2 + 30

= 37.2




In [17]:
df_fe['Engagement_Score'] = (
    df_fe['Visits_Per_Month'] * 0.4 +
    df_fe['Online_Orders'] * 0.4 +
    df_fe['Time_on_Website_Min'] * 0.2
)

In [18]:
df_fe

,Customer_ID,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio,Avg_Spend_Per_Visit,Online_Order_Ratio,Customer_Value,Engagement_Score
0,C0001,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6,0.104498,2275.375000,1.000000,17432,64.2
1,C0002,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6,0.154454,3353.428571,0.857143,22542,63.0
2,C0003,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7,0.359505,12920.142857,1.000000,91063,18.0
3,C0004,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1,0.001846,500.000000,0.000000,0,58.8
4,C0005,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5,0.061268,662.000000,0.200000,1807,28.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8,0.899998,4184.000000,1.000000,61810,65.8
996,C0997,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8,0.824816,9381.250000,0.750000,38901,26.8
997,C0998,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9,0.241858,3103.250000,1.250000,25730,37.2
998,C0999,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6,0.021901,6350.000000,2.000000,7326,20.8


## 4. Preprocessing: Encoding + Scaling

Unsupervised ML algorithms usually numerical input require karte hain. Isliye categorical columns encode aur numerical columns scale kiye jayenge.

In [19]:
features = df_fe.drop(columns=['Customer_ID'])
features

,Age,Gender,City,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio,Avg_Spend_Per_Visit,Online_Order_Ratio,Customer_Value,Engagement_Score
0,58,Male,Karachi,174194,18203,8,8,2179,289,0.09,3.6,0.104498,2275.375000,1.000000,17432,64.2
1,19,Male,Lahore,151981,23474,7,6,3757,289,0.42,3.6,0.154454,3353.428571,0.857143,22542,63.0
2,35,Male,Lahore,251571,90441,7,7,13009,62,0.09,2.7,0.359505,12920.142857,1.000000,91063,18.0
3,40,Female,Karachi,270870,500,1,0,6701,292,0.29,4.1,0.001846,500.000000,0.000000,0,58.8
4,41,Male,Karachi,54025,3310,5,1,1807,129,0.87,3.5,0.061268,662.000000,0.200000,1807,28.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,18,Male,Peshawar,46489,41840,10,10,6181,289,0.01,3.8,0.899998,4184.000000,1.000000,61810,65.8
996,34,Male,Lahore,45495,37525,4,3,12967,120,0.87,2.8,0.824816,9381.250000,0.750000,38901,26.8
997,31,Male,Islamabad,102647,24826,8,10,2573,150,0.48,4.9,0.241858,3103.250000,1.250000,25730,37.2
998,40,Female,Faisalabad,289942,6350,1,2,3663,98,0.20,4.6,0.021901,6350.000000,2.000000,7326,20.8


In [20]:
categorical_cols = ['Gender', 'City']
numerical_cols = [col for col in features.columns if col not in categorical_cols]


In [21]:
categorical_cols

['Gender', 'City']

In [22]:
numerical_cols

['Age',
 'Annual_Income',
 'Monthly_Spending',
 'Visits_Per_Month',
 'Online_Orders',
 'Avg_Order_Value',
 'Time_on_Website_Min',
 'Discount_Usage',
 'Satisfaction_Score',
 'Spending_Ratio',
 'Avg_Spend_Per_Visit',
 'Online_Order_Ratio',
 'Customer_Value',
 'Engagement_Score']

In [23]:
len(numerical_cols)

14

In [24]:

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)

X = preprocessor.fit_transform(features)

In [25]:
X

array([[ 1.16872648,  0.08033732, -0.72581287, ...,  1.        ,
         0.        ,  0.        ],
       [-1.56243373, -0.20440187, -0.58363579, ...,  0.        ,
         1.        ,  0.        ],
       [-0.44195775,  1.07220081,  1.2226955 , ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.72207674, -0.83679384, -0.54716768, ...,  0.        ,
         0.        ,  0.        ],
       [-0.091809  ,  1.56406266, -1.04552921, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.51887523, -1.07651416, -1.20332392, ...,  0.        ,
         0.        ,  0.        ]], shape=(1000, 19))

In [26]:
encoded_cat_cols = preprocessor.named_transformers_['cat']
encoded_cat_cols

,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infrequ

In [27]:
encoded_cat_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
encoded_cat_cols

array(['Gender_Male', 'City_Islamabad', 'City_Karachi', 'City_Lahore',
       'City_Peshawar'], dtype=object)

In [28]:
processed_cols = numerical_cols + list(encoded_cat_cols)
processed_cols

['Age',
 'Annual_Income',
 'Monthly_Spending',
 'Visits_Per_Month',
 'Online_Orders',
 'Avg_Order_Value',
 'Time_on_Website_Min',
 'Discount_Usage',
 'Satisfaction_Score',
 'Spending_Ratio',
 'Avg_Spend_Per_Visit',
 'Online_Order_Ratio',
 'Customer_Value',
 'Engagement_Score',
 'Gender_Male',
 'City_Islamabad',
 'City_Karachi',
 'City_Lahore',
 'City_Peshawar']

In [29]:
encoded_cat_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
processed_cols = numerical_cols + list(encoded_cat_cols)

X_df = pd.DataFrame(X, columns=processed_cols)

In [30]:
X_df

,Age,Annual_Income,Monthly_Spending,Visits_Per_Month,Online_Orders,Avg_Order_Value,Time_on_Website_Min,Discount_Usage,Satisfaction_Score,Spending_Ratio,Avg_Spend_Per_Visit,Online_Order_Ratio,Customer_Value,Engagement_Score,Gender_Male,City_Islamabad,City_Karachi,City_Lahore,City_Peshawar
0,1.168726,0.080337,-0.725813,0.192540,0.529171,-0.464488,1.553947,-1.440606,-0.200393,-0.650015,-0.331746,0.471508,-0.764376,1.586449,1.0,0.0,1.0,0.0,0.0
1,-1.562434,-0.204402,-0.583636,-0.134353,-0.055872,-0.348485,1.553947,-0.288121,-0.200393,-0.514085,-0.254006,0.094711,-0.637256,1.517350,1.0,0.0,0.0,1.0,0.0
2,-0.441958,1.072201,1.222695,-0.134353,0.236650,0.331653,-1.099116,-1.440606,-1.462491,0.043867,0.435860,0.471508,1.067320,-1.073871,1.0,0.0,0.0,1.0,0.0
3,-0.091809,1.319587,-1.203324,-2.095716,-1.810999,-0.132064,1.589010,-0.742130,0.500772,-0.929336,-0.459770,-2.166075,-1.198027,1.275503,0.0,0.0,1.0,0.0,0.0
4,-0.021779,-1.460059,-1.127529,-0.788141,-1.518478,-0.491835,-0.316054,1.283449,-0.340626,-0.767647,-0.448088,-1.638558,-1.153075,-0.486528,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-1.632463,-1.556660,-0.088241,0.846328,1.114213,-0.170291,1.553947,-1.719996,0.080073,1.514568,-0.194113,0.471508,0.339602,1.678581,1.0,0.0,0.0,0.0,1.0
996,-0.511987,-1.569401,-0.204632,-1.115035,-0.933435,0.328566,-0.421241,1.283449,-1.322258,1.309995,0.180666,-0.187887,-0.230298,-0.567143,1.0,0.0,0.0,1.0,0.0
997,-0.722077,-0.836794,-0.547168,0.192540,1.114213,-0.435524,-0.070616,-0.078579,1.622637,-0.276254,-0.272047,1.130904,-0.557949,0.031717,1.0,1.0,0.0,0.0,0.0
998,-0.091809,1.564063,-1.045529,-2.095716,-1.225957,-0.355395,-0.678366,-1.056444,1.201938,-0.874766,-0.037920,3.109092,-1.015780,-0.912640,0.0,0.0,0.0,0.0,0.0
